# FinLLM — Week 2: Logistic Regression

Objective: Train and evaluate a machine learning
model to predict NVIDIA's next-day stock direction.

Inputs:
- 1-day return
- 5-day return
- 20-day volatility
- Relative volume

Target:
1 = Next day's closing price increases
0 = Next day's closing price does not increase

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/remythomasvo/FinLLM/main/data/nvda_features.csv"

df = pd.read_csv(url, index_col="Date", parse_dates=True)

print(df.head())
print(df.shape)
print(df.columns)

               Close      High       Low      Open     Volume  return_1day  \
Date                                                                         
2016-02-02  0.682858  0.709384  0.677991  0.705734  341908000    -0.042321   
2016-02-03  0.686265  0.692349  0.670447  0.688212  326772000     0.004989   
2016-02-04  0.686508  0.702083  0.679694  0.687238  348140000     0.000355   
2016-02-05  0.643191  0.689429  0.641244  0.681884  600428000    -0.063098   
2016-02-08  0.613745  0.636620  0.604741  0.633457  608852000    -0.045781   

            return_5day  volatility_20d   rel_vol  NextDayClose  target  
Date                                                                     
2016-02-02    -0.022300        0.025690  0.812838      0.686265     1.0  
2016-02-03    -0.005642        0.025288  0.792252      0.686508     1.0  
2016-02-04     0.005704        0.024025  0.854542      0.643191     0.0  
2016-02-05    -0.097644        0.026260  1.481968      0.613745     0.0  
2016-02-0

In [21]:

features = [ "return_1day", "return_5day", "volatility_20d", "rel_vol"]

X = df[features]
y = df["target"].astype(int)

print("Inputs:", X.shape)
print("Outputs:", y.shape)

Inputs: (2708, 4)
Outputs: (2708,)


#Data Division

70% Training / 15% Validation / 15% Testing

In [22]:
# data sorted by date
df = df.sort_index()

# calculate split training positions
train_end = int(len(df)*0.70)
valid_end = int(len(df)*0.85)

# divide into three periods
train = df.iloc[:train_end].copy()
valid = df.iloc[train_end:valid_end].copy()
test = df.iloc[valid_end:].copy()

# remove last training row to avoid using first validation day close as a label
train = train.iloc[:-1]

# same for test boundary
valid = valid.iloc[:-1]

print("Training:", train.shape)
print("Validation:", valid.shape)
print("Testing:", test.shape)

Training: (1894, 11)
Validation: (405, 11)
Testing: (407, 11)


In [24]:
from sklearn.preprocessing import StandardScaler

# create scaler
scaler = StandardScaler()

# learn mean and SD from training data only
X_train = scaler.fit_transform(train[features])

# apply same scaling to validation data
X_valid = scaler.transform(valid[features])

# apply same scaling to testing data
X_test = scaler.transform(test[features])

# separate target labels
y_train = train["target"].astype(int)
y_valid = valid["target"].astype(int)
y_test = test["target"].astype(int)

print("Training features:", X_train.shape)
print("Validation features:", X_valid.shape)
print("Testing features:", X_test.shape)

Training features: (1894, 4)
Validation features: (405, 4)
Testing features: (407, 4)


In [26]:
print("Means:", X_train.mean(axis=0))
print("Standard Deviations:", X_train.std(axis=0))

Means: [ 9.37886399e-18 -1.87577280e-18  2.25092736e-16  3.04813080e-16]
Standard Deviations: [1. 1. 1. 1.]


In [28]:
from sklearn.linear_model import LogisticRegression

In [30]:
# create logistic regression model
model = LogisticRegression(max_iter=1000)

# train using the scaled training features and labels
model.fit(X_train, y_train)

print("Model successfully trained!")

Model successfully trained!


In [33]:
for feature, coefficient in zip(features, model.coef_[0]):
  print(feature, ":", round(coefficient, 4))

print("Intercept:", model.intercept_[0])

return_1day : -0.1193
return_5day : 0.0188
volatility_20d : -0.046
rel_vol : 0.0916
Intercept: 0.1811315493282458


In [35]:
# predict whether NVDA goes up or down
y_pred = model.predict(X_valid)

# predict the probability that NVDA goes up
y_prob = model.predict_proba(X_valid)[:, 1]

# display first 10 predictions
print("Predictions:", y_pred[:10])
print("Probabilities:", y_prob[:10])
print("Actual outcomes:", y_valid.iloc[:10].values)

Predictions: [1 1 1 1 1 0 1 1 1 1]
Probabilities: [0.50480585 0.56663607 0.56558284 0.55204835 0.57042873 0.49547865
 0.60076643 0.54756718 0.60665425 0.60154366]
Actual outcomes: [1 0 0 0 1 0 1 1 0 1]


In [36]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_valid, y_pred)

print("Validation accuracy:", round(accuracy * 100, 2), "%")

Validation accuracy: 55.31 %


In [38]:
import numpy as np

# predict that NVDA goes up every day
baseline_pred = np.ones(len(y_valid), dtype=int)

# calculate accuracy
baseline_accuracy = accuracy_score(y_valid, baseline_pred)

print("Logistic regression:", round(accuracy * 100, 2), "%")
print("Always-up baseline:", round(baseline_accuracy * 100, 2), "%")

Logistic regression: 55.31 %
Always-up baseline: 55.56 %


In [39]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion matrix:")
print(confusion_matrix(y_valid, y_pred))

print("/nClassification Report")
print(classification_report(y_valid, y_pred, zero_division=0))

Confusion matrix:
[[ 16 164]
 [ 17 208]]
/nClassification Report
              precision    recall  f1-score   support

           0       0.48      0.09      0.15       180
           1       0.56      0.92      0.70       225

    accuracy                           0.55       405
   macro avg       0.52      0.51      0.42       405
weighted avg       0.53      0.55      0.45       405



In [40]:
from sklearn.metrics import f1_score

model_f1 = f1_score(y_valid, y_pred, average="macro")

baseline_f1 = f1_score(y_valid, baseline_pred, average="macro", zero_division=0)

print("Logistic regression macro-F1", round(model_f1, 4))
print("Always-up macro-F1", round(baseline_f1, 4))

Logistic regression macro-F1 0.4235
Always-up macro-F1 0.3571


## FinLLM — Initial Model Findings

**Model:** Logistic Regression

**Features:** Daily return, five-day momentum, rolling volatility, relative volume

**Validation accuracy:** 55.31%

**Always-up baseline:** 55.56%

**Macro-F1:** Approximately 0.42

**Finding:** Logistic regression did not outperform the always-up baseline on accuracy and showed poor recall for downward movements.

**Next experiment:** Evaluate nonlinear models such as Random Forest and XGBoost using the same chronological data split.